# TrainRef3D v1.0 — one model, one target

Build a MONAI 3D U-Net research baseline from a TrainRef3D Dataset ZIP. Select **Runtime → Change runtime type → T4 GPU**. No automatic uploads: run the upload cell yourself.

DICOM headers are excluded from standard Training ZIPs, but images and names may remain identifiable. Confirm your institution permits upload to Google Colab. Internal validation Dice does not establish clinical validity or generalizability. One-case runs are resubstitution smoke tests, not held-out validation.

Use independently annotated cases; do not count augmented copies of one subject as independent cases. Use InferRef3D and SegRef3D Lite's Custom Model panel for source-bound inference, prediction review, and correction.

In [ ]:
# Defaults are intended for a Colab T4. Advanced settings are optional.
EPOCHS = 100
LEARNING_RATE = 1e-4
PATCH_SIZE = (96, 96, 96)
BATCH_SIZE = 1
NUM_WORKERS = 2
RANDOM_SEED = 42
EARLY_STOPPING_PATIENCE = 20
# For reproducibility after publication, replace main with the tested commit SHA.
# Before publication: upload trainref3d_backend.py via Colab's Files sidebar, then use 'local'.
BACKEND_REF = 'main'


## 1. Set up the runtime

Installs the tested PyTorch 2.8 / MONAI 1.5.1 baseline and downloads this repository's training backend (code only). The backend source hash and library versions are stored in the model manifest. Installing matching torchvision/torchaudio avoids conflicts with Colab's preinstalled packages. Setup can take several minutes.

In [ ]:
%pip -q install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 monai==1.5.1 nibabel==5.3.2 scipy==1.18.1 matplotlib==3.10.6
from pathlib import Path
from urllib.request import urlopen
import hashlib
import torch
assert torch.__version__.split('+')[0] == '2.8.0', 'Restart the runtime after installation, then run all cells again.'
assert torch.cuda.is_available(), 'Select a T4 GPU: Runtime > Change runtime type. CPU is only supported by the separate tiny smoke test.'
backend_path = Path('/content/trainref3d_backend.py')
if BACKEND_REF == 'local':
    assert backend_path.is_file(), 'Upload the reviewed trainref3d_backend.py via the Colab Files sidebar first.'
    backend_source = backend_path.read_bytes()
else:
    backend_url = f'https://raw.githubusercontent.com/SatoruMuro/SegRef3D/{BACKEND_REF}/ColabNotebooks/trainref3d_backend.py'
    with urlopen(backend_url, timeout=60) as response:
        backend_source = response.read()
    backend_path.write_bytes(backend_source)
print('Backend SHA256:', hashlib.sha256(backend_source).hexdigest())
import trainref3d_backend as tr
import importlib
tr = importlib.reload(tr)
print('GPU:', torch.cuda.get_device_name(0))


## 2. Explicitly upload one Dataset ZIP

Create it at [TrainRef3D](https://satorumuro.github.io/SegRef3D/train-web/). Upload goes to **your Google Colab runtime**, not a SegRef3D server. This cell asks for privacy approval before opening the file chooser.

In [ ]:
from google.colab import files
assert input('I am authorized to upload these images to Google Colab (type YES): ').strip() == 'YES', 'Upload cancelled.'
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one TrainRef3D Dataset ZIP.'
upload_name, upload_bytes = next(iter(uploaded.items()))
assert upload_name.lower().endswith('.zip'), 'Expected a Dataset ZIP.'
assert len(upload_bytes) <= tr.MAX_DATASET_BYTES, 'Dataset exceeds the 1 GiB safety limit.'
dataset_path = Path('/content/trainref3d_uploaded_dataset.zip')
dataset_path.write_bytes(upload_bytes)
del uploaded, upload_bytes


## 3. Validate and review the dataset

Read the target, negative cases, RAS-axis median spacing and exact case-level split. No source image values or original labelmaps are modified. Other Obj IDs become background only in temporary binary labels. CT/MRI modality is not guessed from the NIfTI filename; scalar images use per-volume robust clipping and normalization rather than a fixed HU window.

In [ ]:
dataset = tr.prepare_dataset(dataset_path, '/content/trainref3d_work', seed=RANDOM_SEED)


## 4. Train and validate

Patch-based training; full volumes stay on CPU for sliding-window validation. Very large resampled volumes are rejected by a memory guard. Best validation checkpoint and epoch history are saved under the printed run directory. Save output before Colab disconnects.

Validation selects the best epoch on the same internal split, so the displayed Dice is not an independent test-set estimate.

In [ ]:
config = tr.TrainingConfig(epochs=EPOCHS, learning_rate=LEARNING_RATE, patch_size=PATCH_SIZE,
                          batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, random_seed=RANDOM_SEED,
                          patience=EARLY_STOPPING_PATIENCE)
result = tr.train(dataset, '/content/trainref3d_output', config)
tr.plot_history(result)
print('Model ZIP:', result['archive'])


## 5. Download the trained model

Contains best weights, architecture/preprocessing/target/split metadata, epoch history and per-validation-case Dice. No images are added to this model ZIP, but target names and weights still deserve appropriate data governance. This is not a clinically validated model.

In [ ]:
files.download(result['archive'])
